# Leave-One-Subject-Out (LOSO) Evaluation

Self-contained Colab notebook for 34-fold LOSO cross-validation.

**What it does:**
1. Loads cached windows from Google Drive
2. Extracts 84-D handcrafted features
3. Runs 34-fold LOSO for classical models (fast, ~1 hour CPU)
4. Optionally runs LOSO for CNN watch (A1000: ~6-8 hours GPU)

**Setup:**
- Upload `artifacts/windows/` to `MyDrive/cs286-project/artifacts/windows/`
- Runtime → Change runtime type → A1000 GPU
- Run all cells

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration
WINDOWS_ROOT = '/content/drive/MyDrive/cs286-project/artifacts/windows'
OUTPUT_ROOT  = '/content/drive/MyDrive/cs286-project-runs/loso'

SEED = 42
BATCH_SIZE = 512       # A1000 can handle larger batches
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 40
PATIENCE = 7
DROPOUT = 0.2

# A1000 optimizations
USE_AMP = True         # Mixed precision training (significant speedup on A1000)

# Set to True to run CNN watch LOSO (~6-8 hours on A1000)
RUN_CNN_LOSO = True

print('Output root:', OUTPUT_ROOT)

In [ ]:
# Imports
import json, csv, math, random, pickle, time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np

# sklearn for classical LOSO
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

# torch for CNN LOSO
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def detect_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = detect_device()
print('Device:', device)

In [ ]:
# Data loading
window_root = Path(WINDOWS_ROOT)
manifest = json.loads((window_root / 'manifest.json').read_text())
label_to_index = manifest['label_to_index']
index_to_label = {i: l for l, i in label_to_index.items()}
num_classes = len(label_to_index)
print('Classes:', num_classes, list(label_to_index.keys()))

def read_metadata_rows(split: str) -> List[dict]:
    path = window_root / split / 'metadata.csv'
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        return [r for r in reader]

def read_window_payloads(split: str) -> Dict[int, list]:
    payloads = {}
    for chunk_path in sorted((window_root / split).glob('data_chunk_*.pkl')):
        with chunk_path.open('rb') as f:
            records = pickle.load(f)
        for r in records:
            payloads[int(r['window_id'])] = r['x_fusion']
    return payloads

# Load everything once
all_rows = []
all_payloads = {}
for split in ('train', 'val', 'test'):
    rows = read_metadata_rows(split)
    payloads = read_window_payloads(split)
    for r in rows:
        r['split'] = split
    all_rows.extend(rows)
    all_payloads.update(payloads)

print(f'Total windows: {len(all_rows)}')
subject_ids = sorted({int(r['subject_id']) for r in all_rows})
print(f'Subjects: {len(subject_ids)} ({subject_ids[0]}-{subject_ids[-1]})')

In [ ]:
# Feature extraction (same 84-D as train_classical.py)
TARGET_HZ = 20
WINDOW_LEN = 60

def _mean(ch): return float(np.mean(ch))
def _std(ch): return float(np.std(ch))
def _rms(ch): return float(np.sqrt(np.mean(ch ** 2)))
def _zcr(ch):
    if len(ch) < 2: return 0.0
    return float(np.sum(np.diff(np.sign(ch)) != 0) / (len(ch) - 1))
def _dom_freq(ch):
    freqs = np.fft.rfftfreq(len(ch), d=1.0 / TARGET_HZ)
    mags = np.abs(np.fft.rfft(ch))
    if len(mags) <= 1: return 0.0
    peak_bin = int(np.argmax(mags[1:])) + 1
    return float(freqs[peak_bin])
def _spectral_energy(ch):
    mags = np.abs(np.fft.rfft(ch))
    return float(np.sum(mags ** 2) / len(ch))
def _spectral_entropy(ch):
    mags = np.abs(np.fft.rfft(ch))
    power = mags ** 2
    total = power.sum()
    if total < 1e-12: return 0.0
    p = power / total
    return float(-np.sum(p * np.log2(p + 1e-12)))

def _channel_features(ch):
    return [_mean(ch), _std(ch), _rms(ch), _zcr(ch), _dom_freq(ch), _spectral_energy(ch), _spectral_entropy(ch)]

def extract_window_features(x):
    feats = []
    for ch_idx in range(12):
        feats.extend(_channel_features(x[ch_idx]))
    return np.array(feats, dtype=np.float32)

def extract_features_for_rows(rows, payloads):
    X = np.empty((len(rows), 84), dtype=np.float32)
    y = np.empty(len(rows), dtype=np.int64)
    subjects = np.empty(len(rows), dtype=np.int64)
    for i, r in enumerate(rows):
        x = payloads[int(r['window_id'])]
        X[i] = extract_window_features(x)
        y[i] = int(r['label_idx'])
        subjects[i] = int(r['subject_id'])
    return X, y, subjects

print('Extracting features for all', len(all_rows), 'windows...')
X_all, y_all, s_all = extract_features_for_rows(all_rows, all_payloads)
print('Done. X shape:', X_all.shape)

In [ ]:
# Classical model definitions
def build_classical_models():
    models = {
        'dt': DecisionTreeClassifier(max_depth=20, random_state=SEED),
        'rf': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
        'nb': GaussianNB(),
        'svm': CalibratedClassifierCV(LinearSVC(C=0.1, max_iter=2000, random_state=SEED)),
        'ada': AdaBoostClassifier(n_estimators=100, random_state=SEED),
    }
    try:
        from xgboost import XGBClassifier
        models['xgb'] = XGBClassifier(n_estimators=200, tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        models['xgb'] = HistGradientBoostingClassifier(max_iter=200, random_state=SEED)
    return models

MODEL_NAMES = {
    'dt': 'Decision Tree', 'rf': 'Random Forest', 'nb': 'Gaussian NB',
    'svm': 'Linear SVM', 'ada': 'AdaBoost', 'xgb': 'XGBoost / HGB'
}

print('Classical models ready.')

In [ ]:
# Run classical LOSO
output_root = Path(OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

models_template = build_classical_models()
per_fold_results = []

for held_out in subject_ids:
    print(f'\nFold {held_out} / subjects {subject_ids[0]}-{subject_ids[-1]}')
    train_mask = s_all != held_out
    test_mask = s_all == held_out
    
    X_train, y_train = X_all[train_mask], y_all[train_mask]
    X_test, y_test = X_all[test_mask], y_all[test_mask]
    
    fold_result = {'held_out_subject': int(held_out), 'n_train': int(train_mask.sum()), 'n_test': int(test_mask.sum())}
    
    for key, clf_template in models_template.items():
        clf = clone(clf_template)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        acc = float(accuracy_score(y_test, y_pred))
        macro_f1 = float(f1_score(y_test, y_pred, average='macro', zero_division=0))
        fold_result[key] = {'accuracy': acc, 'macro_f1': macro_f1}
        print(f'  {MODEL_NAMES[key]:<16} acc={acc:.4f} f1={macro_f1:.4f}')
    
    per_fold_results.append(fold_result)

print('\nClassical LOSO complete!')

In [ ]:
# Aggregate classical LOSO results
aggregate = {}
for key in models_template:
    accs = [fold[key]['accuracy'] for fold in per_fold_results]
    f1s = [fold[key]['macro_f1'] for fold in per_fold_results]
    aggregate[key] = {
        'display_name': MODEL_NAMES[key],
        'mean_accuracy': float(np.mean(accs)),
        'std_accuracy': float(np.std(accs)),
        'mean_macro_f1': float(np.mean(f1s)),
        'std_macro_f1': float(np.std(f1s)),
        'min_accuracy': float(np.min(accs)),
        'max_accuracy': float(np.max(accs)),
        'per_fold_accuracy': accs,
        'per_fold_macro_f1': f1s,
    }

# Save JSON
results_payload = {
    'protocol': 'leave-one-subject-out',
    'n_folds': len(subject_ids),
    'n_total_samples': int(len(X_all)),
    'subject_ids': subject_ids,
    'per_fold': per_fold_results,
    'aggregate': aggregate,
}
json_path = output_root / 'classical_loso_results.json'
json_path.write_text(json.dumps(results_payload, indent=2), encoding='utf-8')
print(f'Saved → {json_path}')

# Save CSV summary
csv_path = output_root / 'classical_loso_summary.csv'
with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Model', 'Mean Acc', 'Std Acc', 'Mean F1', 'Std F1', 'Min Acc', 'Max Acc'])
    for key in models_template:
        a = aggregate[key]
        writer.writerow([
            a['display_name'],
            f"{a['mean_accuracy']:.4f}",
            f"{a['std_accuracy']:.4f}",
            f"{a['mean_macro_f1']:.4f}",
            f"{a['std_macro_f1']:.4f}",
            f"{a['min_accuracy']:.4f}",
            f"{a['max_accuracy']:.4f}",
        ])
print(f'Saved → {csv_path}')

# Print summary
print('\n' + '=' * 60)
print('  CLASSICAL LOSO SUMMARY (34 folds)')
print('=' * 60)
print(f"{'Model':<16} {'Mean Acc':>10} {'Std Acc':>10} {'Mean F1':>10} {'Std F1':>10}")
print('-' * 60)
for key in models_template:
    a = aggregate[key]
    print(f"{a['display_name']:<16} {a['mean_accuracy']:>10.4f} {a['std_accuracy']:>10.4f} {a['mean_macro_f1']:>10.4f} {a['std_macro_f1']:>10.4f}")
print('=' * 60)

---

# Optional: CNN Watch LOSO

**WARNING:** This takes ~6-8 hours on A1000. Set `RUN_CNN_LOSO = True` above.

In [ ]:
# CNN model definition (watch-only)
class TimeSeriesEncoder(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, 5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
    def forward(self, x):
        return self.features(x).squeeze(-1)

class CNNWatchModel(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.encoder = TimeSeriesEncoder(6)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

print('CNN model ready.')

In [ ]:
# A1000 optimized CNN training with mixed precision
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler() if USE_AMP and torch.cuda.is_available() else None

def build_watch_arrays(rows, payloads):
    x = np.stack([payloads[int(r['window_id'])][6:12] for r in rows]).astype(np.float32)
    y = np.asarray([int(r['label_idx']) for r in rows], dtype=np.int64)
    s = np.asarray([int(r['subject_id']) for r in rows], dtype=np.int64)
    return x, y, s

def to_loader(x, y, batch_size, shuffle):
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                      batch_size=batch_size, shuffle=shuffle, num_workers=0)

def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        
        if USE_AMP and scaler is not None:
            with autocast():
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        total_loss += float(loss.item()) * len(xb)
        total += len(xb)
    return total_loss / max(1, total)

def evaluate_cnn(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if USE_AMP:
                with autocast():
                    logits = model(xb)
                    loss = loss_fn(logits, yb)
            else:
                logits = model(xb)
                loss = loss_fn(logits, yb)
            total_loss += float(loss.item()) * len(xb)
            all_true.extend(yb.cpu().tolist())
            all_pred.extend(logits.argmax(1).cpu().tolist())
    acc = sum(1 for t, p in zip(all_true, all_pred) if t == p) / len(all_true)
    return {'accuracy': float(acc), 'loss': total_loss / len(all_true)}

print(f'CNN utilities ready. AMP={USE_AMP}, Batch size={BATCH_SIZE}')

In [ ]:
# Run CNN watch LOSO
if RUN_CNN_LOSO:
    cnn_loso_results = []
    start_time = time.time()
    
    for fold_idx, held_out in enumerate(subject_ids):
        elapsed = time.time() - start_time
        eta = (elapsed / max(1, fold_idx)) * (len(subject_ids) - fold_idx) if fold_idx > 0 else 0
        print(f'\n=== CNN LOSO Fold {held_out} ({fold_idx+1}/{len(subject_ids)}) — ETA: {eta/3600:.1f}h ===')
        
        train_rows = [r for r in all_rows if int(r['subject_id']) != held_out]
        test_rows = [r for r in all_rows if int(r['subject_id']) == held_out]
        
        train_subjects = sorted({int(r['subject_id']) for r in train_rows})
        val_subjects = train_subjects[-4:] if len(train_subjects) >= 4 else train_subjects[:1]
        val_rows_fold = [r for r in train_rows if int(r['subject_id']) in val_subjects]
        final_train_rows = [r for r in train_rows if int(r['subject_id']) not in val_subjects]
        
        x_tr, y_tr, _ = build_watch_arrays(final_train_rows, all_payloads)
        x_va, y_va, _ = build_watch_arrays(val_rows_fold, all_payloads)
        x_te, y_te, _ = build_watch_arrays(test_rows, all_payloads)
        
        train_loader = to_loader(x_tr, y_tr, BATCH_SIZE, shuffle=True)
        val_loader = to_loader(x_va, y_va, BATCH_SIZE, shuffle=False)
        test_loader = to_loader(x_te, y_te, BATCH_SIZE, shuffle=False)
        
        seed_everything(SEED)
        model = CNNWatchModel(num_classes, DROPOUT).to(device)
        
        class_counts = np.bincount(y_tr, minlength=num_classes)
        class_weights = torch.tensor(1.0 / (class_counts.astype(np.float32) + 1.0), dtype=torch.float32)
        class_weights = class_weights / class_weights.sum() * num_classes
        class_weights = class_weights.to(device)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
        
        best_f1 = -1.0
        best_state = None
        epochs_no_improve = 0
        
        for epoch in range(1, MAX_EPOCHS + 1):
            tr_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
            val_m = evaluate_cnn(model, val_loader, loss_fn, device)
            
            if epoch % 5 == 0 or epoch == 1:
                print(f'  e={epoch:02d} loss={tr_loss:.4f} val_acc={val_m["accuracy"]:.4f}')
            
            if val_m['accuracy'] > best_f1:
                best_f1 = val_m['accuracy']
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE:
                    print(f'  early stopping at epoch {epoch}')
                    break
            scheduler.step(val_m['accuracy'])
        
        model.load_state_dict(best_state)
        test_m = evaluate_cnn(model, test_loader, loss_fn, device)
        
        print(f'  TEST: acc={test_m["accuracy"]:.4f}')
        cnn_loso_results.append({
            'held_out_subject': int(held_out),
            'accuracy': test_m['accuracy'],
            'loss': test_m['loss']
        })
        
        # Save intermediate results every 5 folds
        if (fold_idx + 1) % 5 == 0:
            cnn_json_path = output_root / 'cnn_watch_loso_results_partial.json'
            cnn_json_path.write_text(json.dumps({
                'protocol': 'leave-one-subject-out',
                'model': 'CNN watch',
                'completed_folds': fold_idx + 1,
                'results': cnn_loso_results
            }, indent=2), encoding='utf-8')
            print(f'  Saved partial results → {cnn_json_path}')
    
    # Save final CNN LOSO results
    cnn_accs = [r['accuracy'] for r in cnn_loso_results]
    cnn_payload = {
        'protocol': 'leave-one-subject-out',
        'model': 'CNN watch',
        'n_folds': len(subject_ids),
        'mean_accuracy': float(np.mean(cnn_accs)),
        'std_accuracy': float(np.std(cnn_accs)),
        'per_fold': cnn_loso_results
    }
    cnn_json_path = output_root / 'cnn_watch_loso_results.json'
    cnn_json_path.write_text(json.dumps(cnn_payload, indent=2), encoding='utf-8')
    print(f'\nCNN LOSO saved → {cnn_json_path}')
    print(f'CNN LOSO mean accuracy: {np.mean(cnn_accs):.4f} ± {np.std(cnn_accs):.4f}')
else:
    print('CNN LOSO skipped. Set RUN_CNN_LOSO = True to enable.')

---

# Download Results

Results are saved to your Google Drive under `cs286-project-runs/loso/`.

Key files:
- `classical_loso_results.json` — full per-fold results
- `classical_loso_summary.csv` — summary table
- `cnn_watch_loso_results.json` — CNN LOSO results (if run)